<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/14_Deep_Learning/04_Model_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Phase 15: Production Engineering
#### Part 01: Post-Training Quantization

To deploy our neural networks to general edge servers and production APIs, we must compress them. Post-training quantization converts the standard 32-bit floating-point weights into highly optimized 8-bit integers.

This transformation drastically reduces the memory footprint and accelerates HTTP API inference latency, allowing our models to process live environmental data at scale with virtually no loss in mathematical accuracy. We will convert all three `.keras` models into optimized `.tflite` engines.

In [ ]:
import tensorflow as tf
import os

# Ensure models directory exists
os.makedirs('models', exist_ok=True)

def quantize_model(keras_path, tflite_path):
    print(f"⏳ Converting {keras_path} to optimized TFLite...")

    # 1. Load the original Keras model
    model = tf.keras.models.load_model(keras_path)

    # 2. Initialize the converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 3. Apply dynamic range quantization (compresses weights to 8-bit precision)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # 4. Execute conversion
    tflite_model = converter.convert()

    # 5. Save the production-ready model
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)

    # Calculate and display the compression metrics
    keras_size = os.path.getsize(keras_path) / 1024
    tflite_size = os.path.getsize(tflite_path) / 1024
    print(f"✅ Saved to: {tflite_path}")
    print(f"📉 Size reduction: {keras_size:.1f} KB -> {tflite_size:.1f} KB\n")

# Define the model conversion mapping
models_to_convert = [
    ('models/mlp_tabular_v1.keras', 'models/mlp_tabular_quantized.tflite'),
    ('models/cnn_spatial_vision_v1.keras', 'models/cnn_spatial_quantized.tflite'),
    ('models/lstm_temporal_forecast_v1.keras', 'models/lstm_temporal_quantized.tflite')
]

# Execute the pipeline
for keras_file, tflite_file in models_to_convert:
    if os.path.exists(keras_file):
        quantize_model(keras_file, tflite_file)
    else:
        print(f"⚠️ Warning: '{keras_file}' not found. Make sure all previous notebooks successfully saved their models.")